# 03 · Comparar detectores — OPCIONAL
Requisito: dataset PlantDoc verificado en 00. Es recomendable completar 02;
su ejecución YOLO26 se reutiliza si tiene la misma configuración.
Compara YOLO26, YOLO11 y YOLOv8 sin inventar métricas.
**Selecciona con validación; no toca test**. Usa 04 para la evaluación final.
Si solo necesitas YOLO26, omite este cuaderno.


## Celda 01 · Conectar Drive y localizar o extraer el proyecto

Repetir en cada cuaderno y después de reiniciar el entorno. Sube `colab_yolo26_listo.zip` a Mi unidad: si falta el proyecto, esta celda lo extrae automáticamente. Conserva los datos existentes y no escribe en `runs/`. Para otras rutas, define `IOTTESIS_ROOT` (carpeta que contiene `scripts/`) o `IOTTESIS_ZIP` (archivo ZIP). Ningún cuaderno depende de las variables de otro.


In [4]:
# Celda 01 · Conectar Drive y localizar o extraer el proyecto
"""Self-contained Colab startup, also embedded in the notebooks before imports."""
import importlib
from itertools import islice
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import sys
from zipfile import ZipFile


PROJECT_FILES = (
    'scripts/flujo_colab.py', 'scripts/entrenar_yolo26.py', 'scripts/preparar_datasets_reales.py',
    'requirements-datasets.txt', 'requirements-vision.txt',
)


def iniciar_colab(*, drive_home='/content/drive/MyDrive', content='/content', mount=True):
    """Find a complete project, or install the local bundle without replacing data."""
    drive_home, content = Path(drive_home), Path(content)
    if mount:
        try:
            from google.colab import drive
        except ImportError:
            drive = None
        if drive is not None:
            drive.mount(str(drive_home.parent))

    default_root = drive_home / 'Tesis_IOT_Papa'
    configured = os.environ.get('IOTTESIS_ROOT')
    requested = Path(configured).expanduser().resolve() if configured else None
    # The previous notebook set this default even when the folder did not exist.
    explicit = requested is not None and requested != default_root.resolve()
    bases = [requested] if explicit else [default_root, drive_home / 'IOTtesis',
                                         content / 'IOTtesis', content / 'Tesis_IOT_Papa',
                                         Path.cwd(), *Path.cwd().parents]

    def ready(path):
        return all((path / name).is_file() for name in PROJECT_FILES)

    candidates = list(bases)
    for base in bases:
        candidates.extend(base / name for name in ('Tesis_IOT_Papa', 'IOTtesis'))
        if base.name == 'notebooks':
            candidates.append(base.parent)
    root = next((p.resolve() for p in candidates if ready(p)), None)
    if root is None and not explicit and drive_home.is_dir():
        # Bounded discovery: renamed project folders, and one enclosing folder.
        discovered = set()
        for folder in islice(drive_home.iterdir(), 200):
            if folder.name.startswith('.') or not folder.is_dir():
                continue
            for candidate in (folder, folder / 'Tesis_IOT_Papa', folder / 'IOTtesis'):
                if ready(candidate):
                    discovered.add(candidate.resolve())
        if len(discovered) > 1:
            raise RuntimeError('Hay varios proyectos. Define os.environ["IOTTESIS_ROOT"] con uno de: '
                               + ', '.join(str(p) for p in sorted(discovered)))
        root = next(iter(discovered), None)

    if root is None:
        zip_override = os.environ.get('IOTTESIS_ZIP')
        archives = ([Path(zip_override).expanduser()] if zip_override else
                    [base / 'colab_yolo26_listo.zip' for base in (drive_home, content, Path.cwd())])
        archive = next((p for p in archives if p.is_file()), None)
        if archive is None:
            raise RuntimeError(
                'No se encontró el proyecto completo ni el ZIP.\n'
                'Sube colab_yolo26_listo.zip a Mi unidad y vuelve a ejecutar esta celda.\n'
                f'ZIP esperado: {archives[0]}\n'
                'Si usas otras rutas, define os.environ["IOTTESIS_ROOT"] con la carpeta '
                'que contiene scripts/ y os.environ["IOTTESIS_ZIP"] con el archivo ZIP.\n'
                f'Carpeta de proyecto esperada: {requested or default_root}')
        root = requested if requested is not None else (
            default_root if drive_home.is_dir() else
            (content if content.is_dir() else Path.cwd()) / 'Tesis_IOT_Papa')
        root = root.resolve()
        with ZipFile(archive) as package:
            entries = []
            seen = set()
            for member in package.infolist():
                path = PurePosixPath(member.filename)
                parts = path.parts
                if ('\\' in member.filename or path.is_absolute() or '..' in parts
                        or not parts or parts[0] != 'Tesis_IOT_Papa'
                        or stat.S_ISLNK(member.external_attr >> 16)):
                    raise ValueError('Ruta inválida en el ZIP: ' + member.filename)
                if len(parts) == 1 or member.is_dir():
                    continue
                relative = PurePosixPath(*parts[1:])
                if (relative.parts[0] not in {'scripts', 'notebooks', 'data', 'docs',
                                             'requirements-datasets.txt', 'requirements-vision.txt',
                                             'EMPEZAR_AQUI.txt'}
                        or any(part.startswith('.') for part in relative.parts)
                        or relative.suffix.lower() in {'.pt', '.db', '.sqlite3', '.key', '.pem'}):
                    raise ValueError('Archivo ajeno al paquete de Colab: ' + str(relative))
                target = root / str(relative)
                if relative in seen or not target.resolve().is_relative_to(root):
                    raise ValueError('Ruta repetida o externa en el ZIP: ' + str(relative))
                seen.add(relative)
                entries.append((member, target, relative.parts[0] == 'data'))
            if not all(PurePosixPath(name) in seen for name in PROJECT_FILES):
                raise ValueError('El ZIP no contiene el flujo completo. Usa colab_yolo26_listo.zip actualizado.')
            # Check all paths before writing; never follow existing linked directories.
            for _, target, _ in entries:
                if target.is_symlink() or any(p.is_symlink() for p in target.parents if p != root and p.is_relative_to(root)):
                    raise ValueError('Destino enlazado: ' + str(target))
                if target.exists() and not target.is_file():
                    raise ValueError('El destino no es un archivo: ' + str(target))
            preserved = 0
            for member, target, is_data in entries:
                if is_data and target.exists():
                    preserved += 1
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with package.open(member) as source, target.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
        print('Proyecto extraído desde:', archive)
        if preserved:
            print('Datos existentes conservados:', preserved, '| Verifícalos en 00, celda 4.')

    os.environ['IOTTESIS_ROOT'] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    importlib.invalidate_caches()
    print('Proyecto:', root)
    return root

PROJECT_ROOT = iniciar_colab()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto: /content/drive/MyDrive/Tesis_IOT_Papa


## Celda 02 · Instalar dependencias de esta sesión

Instala visión en la sesión actual. Si ya se había importado otra versión de Ultralytics, reinicia el kernel y vuelve a empezar.


In [5]:
# Celda 02 · Instalar dependencias de esta sesión
%pip install -q -r "{PROJECT_ROOT / 'requirements-vision.txt'}"


## Celda 03 · Comprobar entorno y cargar el flujo

El estado se calcula a partir de datasets y checkpoints guardados; ejecutar una celda no cuenta como entrenamiento completado. En 01–04 selecciona GPU T4. Cambiar el entorno reinicia las variables: repite las celdas 1–3.


In [7]:
# Celda 03 · Comprobar entorno y cargar el flujo
import json
import torch
import ultralytics
from scripts.entrenar_yolo26 import training_plan
from scripts.flujo_colab import (
    inspect_progress, print_progress, run_or_reuse, latest_experiment,
    freeze_detector, final_records, evaluate_once, comparable_detectors, write_summary,
)
if tuple(int(v) for v in ultralytics.__version__.split('.')[:2]) < (8, 4):
    raise RuntimeError('Instala Ultralytics >=8.4.0 y reinicia el kernel.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No hay una GPU CUDA disponible. En Colab selecciona GPU T4 en el tipo de entorno y vuelve a ejecutar las celdas 1, 2 y 3.')
DEVICE = 0
print('GPU:', torch.cuda.get_device_name(DEVICE), '| Dispositivo CUDA:', DEVICE)

print('Ultralytics:', ultralytics.__version__, '| PyTorch:', torch.__version__, '| Dispositivo:', DEVICE)
print_progress(inspect_progress(PROJECT_ROOT))


GPU: Tesla T4 | Dispositivo CUDA: 0
Ultralytics: 8.4.201 | PyTorch: 2.11.0+cu130 | Dispositivo: 0
PROYECTO: /content/drive/MyDrive/Tesis_IOT_Papa
Datos plantvillage: listo | 2152 imágenes
Datos plantdoc: listo | 219 imágenes
01 Clasificador: /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo26n_plantvillage_classify/weights/best.pt
02 Detector: /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo26n_plantdoc_detect/weights/best.pt
03 Comparación: opcional, antes de evaluar test
04 Test final: PENDIENTE
Siguiente paso de la ruta corta: 04_evaluacion_y_exportacion.ipynb
REVISAR: ['/content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolov8n_plantdoc_detect/weights/last.pt'] []
Riesgo real: pendiente de serie temporal y etiquetas reales; demo separada


## Celda 04 · Fijar la comparación

Perfil GPU T4: dispositivo CUDA 0 y lote 8. La celda importa training_plan explícitamente. Restaura imágenes dañadas desde el ZIP compatible y guarda respaldo. Mismas particiones, semilla, lote, épocas, tamaño, SGD y scheduler cosine. No cambies estos parámetros entre familias para la misma comparación.


In [15]:
# Celda 04 · Fijar la comparación
if 'PROJECT_ROOT' not in globals():
    raise RuntimeError('El entorno perdió sus variables. Ejecuta las celdas 1, 2 y 3 de este cuaderno y luego repite la 4.')
import json
from scripts.entrenar_yolo26 import training_plan
from scripts.flujo_colab import (
    inspect_progress, print_progress, run_or_reuse, latest_experiment,
    freeze_detector, final_records, evaluate_once, comparable_detectors, write_summary,
)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No hay una GPU CUDA disponible. En Colab selecciona GPU T4 en el tipo de entorno y vuelve a ejecutar las celdas 1, 2 y 3.')
DEVICE = 0
print('GPU:', torch.cuda.get_device_name(DEVICE), '| Dispositivo CUDA:', DEVICE)
"""Restore damaged prepared files from the matching bundle, keeping backups."""
import hashlib
import json
import os
from pathlib import Path
import shutil
import tempfile
from uuid import uuid4
from zipfile import ZipFile

import yaml
from scripts.preparar_datasets_reales import image_info, verify_dataset


def repair_dataset(root, source, archive=None):
    root = Path(root).resolve()
    folder = {'plantdoc': 'papa_plantdoc_yolo',
              'plantvillage': 'papa_plantvillage_clasificacion'}[source]
    dataset = root / 'data/processed' / folder
    try:
        return verify_dataset(dataset)
    except (OSError, ValueError, KeyError) as original_error:
        candidates = ([Path(archive)] if archive else
                      [Path(os.environ['IOTTESIS_ZIP'])] if os.environ.get('IOTTESIS_ZIP') else
                      [root.parent / 'colab_yolo26_listo.zip',
                       root / 'data/processed/colab_yolo26_listo.zip', Path('/content/colab_yolo26_listo.zip')])
        bundle = next((p for p in candidates if p.is_file()), None)
        if bundle is None or not (dataset / 'manifest.json').is_file():
            raise RuntimeError('El dataset no pasa la verificación. Usa 00 para prepararlo o coloca '
                               'colab_yolo26_listo.zip en Mi unidad para restaurarlo. Detalle: '
                               + str(original_error)) from original_error

    manifest_bytes = (dataset / 'manifest.json').read_bytes()
    manifest = json.loads(manifest_bytes)
    prefix = 'Tesis_IOT_Papa/data/processed/' + folder + '/'
    with tempfile.TemporaryDirectory(prefix='iot-restaurar-') as temporary, ZipFile(bundle) as package:
        staged = Path(temporary)
        bundled = json.loads(package.read(prefix + 'manifest.json'))
        # Excluded-source notes may have been expanded; keep the installed manifest intact.
        if any(manifest.get(k) != bundled.get(k)
               for k in ('source', 'task', 'revision', 'classes', 'records', 'synthetic')):
            raise ValueError('El ZIP contiene otra revisión, partición o anotación del dataset. No se sustituyó ningún archivo.')
        paths = []
        for record in manifest['records']:
            paths.append((record['output_path'], record['pixel_sha256']))
            if record.get('label_path'):
                paths.append((record['label_path'], None))
        if manifest['task'] == 'detect':
            paths.append(('data.yaml', None))
        seen = set()
        for relative, _ in paths:
            path = Path(relative)
            if path.is_absolute() or '..' in path.parts or relative in seen:
                raise ValueError('Ruta inválida o repetida en el manifiesto: ' + relative)
            seen.add(relative)
            target = dataset / path
            if not target.resolve().is_relative_to(dataset.resolve()) or target.is_symlink():
                raise ValueError('Destino externo o enlazado: ' + relative)
            output = staged / path
            output.parent.mkdir(parents=True, exist_ok=True)
            output.write_bytes(package.read(prefix + relative))
        (staged / 'manifest.json').write_bytes(manifest_bytes)
        verify_dataset(staged)  # validate the full replacement before touching Drive

        changed = []
        for relative, expected_pixels in paths:
            current, replacement = dataset / relative, staged / relative
            try:
                if expected_pixels is not None:
                    intact = image_info(current)[1] == expected_pixels
                elif relative == 'data.yaml':
                    old, new = yaml.safe_load(current.read_text()), yaml.safe_load(replacement.read_text())
                    intact = old.get('nc') == new.get('nc') and old.get('names') == new.get('names')
                else:
                    intact = current.read_bytes() == replacement.read_bytes()
            except (OSError, ValueError, AttributeError, yaml.YAMLError):
                intact = False
            if not intact:
                changed.append(relative)
        if not changed:
            # A structural issue must not be disguised as a repaired image.
            return verify_dataset(dataset)

        backup = root / 'data/repair_backups' / uuid4().hex / folder
        backup.mkdir(parents=True)
        # Back up every existing input before replacing the first file.
        for relative in changed:
            current = dataset / relative
            if current.exists():
                saved = backup / relative
                saved.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(current, saved)
        log = {'source': source, 'archive': str(bundle), 'replaced': changed,
               'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(), 'status': 'restoring'}
        (backup / 'repair.json').write_text(json.dumps(log, ensure_ascii=False, indent=2))
        for relative in changed:
            target = dataset / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            pending = target.with_name('.' + target.name + '.restoring-' + uuid4().hex)
            try:
                shutil.copyfile(staged / relative, pending)
                pending.replace(target)
            finally:
                pending.unlink(missing_ok=True)
        result = verify_dataset(dataset)
        log['status'] = 'verified'
        (backup / 'repair.json').write_text(json.dumps(log, ensure_ascii=False, indent=2))
        print('Dataset restaurado:', source, '| Archivos:', len(changed), '| Respaldo:', backup)
        return result

repair_dataset(PROJECT_ROOT, 'plantdoc')

FAMILIES = ['yolo26', 'yolo11']
EPOCHS = 50
BATCH = 8
REENTRENAR = False
PERMITIR_CPU = False
plans = [training_plan(PROJECT_ROOT, 'plantdoc', family, epochs=EPOCHS, batch=BATCH)
         for family in FAMILIES]
print([(p['family'], p['epochs'], p['batch'], p['images']) for p in plans])


GPU: Tesla T4 | Dispositivo CUDA: 0
[('yolo26', 50, 8, 219), ('yolo11', 50, 8, 219)]


## Celda 05 · Entrenar solo las familias que falten

Puede iniciar dos o tres entrenamientos, según lo que ya esté guardado. No vuelve a entrenar YOLO26 de 02 si la configuración coincide.


In [16]:
# Celda 05 · Entrenar solo las familias que falten
records = [run_or_reuse(plan, device=DEVICE, retrain=REENTRENAR, allow_cpu=PERMITIR_CPU)
           for plan in plans]
for record in records:
    print(record['family'], record['validation'], record['best_weights'])


Ya entrenado; reutilizando: /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo26n_plantdoc_detect/weights/best.pt
Ya entrenado; reutilizando: /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo11n_plantdoc_detect/weights/best.pt
yolo26 {'metrics/precision(B)': 0.46896933071887736, 'metrics/recall(B)': 0.7453365715454467, 'metrics/mAP50(B)': 0.6343364049747451, 'metrics/mAP50-95(B)': 0.46993872016654403, 'fitness': 0.46993872016654403} /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo26n_plantdoc_detect/weights/best.pt
yolo11 {'metrics/precision(B)': 0.5566497474851205, 'metrics/recall(B)': 0.5807532831131691, 'metrics/mAP50(B)': 0.5917793372665938, 'metrics/mAP50-95(B)': 0.3981353638936549, 'fitness': 0.3981353638936549} /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo11n_plantdoc_detect/weights/best.pt


## Celda 06 · Seleccionar por validación y guardar la decisión

No se declara un ganador anticipadamente y no se consulta test para elegirlo. La referencia se guarda con una ruta portable para abrir 04 en otra sesión.


In [17]:
# Celda 06 · Seleccionar por validación y guardar la decisión
winner = freeze_detector(records, PROJECT_ROOT)
print('Seleccionado por mAP50-95 de validación:', winner['family'])
print('Pesos:', winner['best_weights'])
print('Selección:', PROJECT_ROOT / 'runs/datasets_reales/seleccion_por_validacion.json')


Seleccionado por mAP50-95 de validación: yolo26
Pesos: /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/yolo26n_plantdoc_detect/weights/best.pt
Selección: /content/drive/MyDrive/Tesis_IOT_Papa/runs/datasets_reales/seleccion_por_validacion.json


## Siguiente paso
Abre **04_evaluacion_y_exportacion.ipynb**. Allí se cargarán el clasificador
de 01 y el detector seleccionado aquí. Si deseas reportar todas las familias
en test, puedes evaluarlas una vez después de congelar esta selección;
no vuelvas a elegir el ganador a partir de esos resultados.
